# 03 — Embeddings & Vector Store

**Input:** Notebook 02 production chunks  
**Output:** dense embeddings + FAISS vector index + metadata

Pipeline:

`Chunk → tokenizer → token IDs → embedding model → dense vector → normalization → FAISS`

In [9]:
from pathlib import Path
import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("Notebook 03 ready")

Notebook 03 ready


In [10]:
PROJECT_ROOT = Path.cwd().parent
CHUNKS_DIR = PROJECT_ROOT / "data" / "chunks"
VECTOR_DIR = PROJECT_ROOT / "data" / "vector_store"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("Chunks:", CHUNKS_DIR)
print("Vector store:", VECTOR_DIR)

Chunks: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\chunks
Vector store: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\vector_store


In [11]:
# Embedding model
# BGE is a strong general-purpose retrieval embedding model.
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

BATCH_SIZE = 32
NORMALIZE_EMBEDDINGS = True

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Model:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Max sequence length:", model.max_seq_length)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6218.69it/s]


Model: BAAI/bge-small-en-v1.5
Embedding dimension: 384
Max sequence length: 512


C:\Users\MD RAJA\AppData\Local\Temp\ipykernel_24072\3746843613.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [12]:
# Load Notebook 02 chunk files
chunk_files = sorted(
    p for p in CHUNKS_DIR.glob("*_chunks.json")
    if p.name != "chunk_manifest.json"
)

assert chunk_files, f"No chunk files found in {CHUNKS_DIR}"

all_chunks = []

for path in chunk_files:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    all_chunks.extend(data.get("chunks", []))

print("Chunk files:", len(chunk_files))
print("Total chunks:", len(all_chunks))

Chunk files: 11
Total chunks: 1141


In [13]:
# Basic chunk QA before embedding

texts = [c.get("retrieval_text") or c.get("text") or "" for c in all_chunks]

empty_chunks = [
    c for c, t in zip(all_chunks, texts)
    if not t.strip()
]

duplicate_ids = len(all_chunks) - len({
    c["chunk_id"] for c in all_chunks
})

print("Empty texts:", len(empty_chunks))
print("Duplicate chunk IDs:", duplicate_ids)

# Inspect empty chunks
for c in empty_chunks:
    print(
        c["chunk_id"],
        "| type:", c.get("chunk_type"),
        "| content_types:", c.get("content_types"),
        "| section:", c.get("section_path")
    )

assert duplicate_ids == 0

Empty texts: 8
Duplicate chunk IDs: 0
chunk_eee2b795110d64a6 | type: figure_evidence | content_types: ['figure'] | section: ['Figure 1 Model timeline']
chunk_51353ebd7a969354 | type: figure_evidence | content_types: ['figure'] | section: ['Figure 1 Model timeline']
chunk_78f6c926f44870fd | type: figure_evidence | content_types: ['figure'] | section: ['Price efficiency as a function of quantitative investing by firm B (both firms have quant departments)']
chunk_c2d5c622c48b84c2 | type: figure_evidence | content_types: ['figure'] | section: ['Price efficiency as a function of quantitative investing level by both firms γ1 and γ2']
chunk_09f9b2c55130cf38 | type: figure_evidence | content_types: ['figure'] | section: ['Trading aggressiveness as a function of quantitative investing with partially myopic quant department of firm B (firm A is fully discretionary)']
chunk_fbd157ad6a8992bc | type: figure_evidence | content_types: ['figure'] | section: ['Trading aggressiveness as a function of qu

In [15]:
# Final token-limit check using the ACTUAL embedding model tokenizer.
#
# This is the important distinction:
# - Notebook 02 performs production chunking.
# - Notebook 03 verifies that chunks fit the actual embedding model.
#
# Tokenization is always required before embedding.

tokenizer = model.tokenizer

def count_model_tokens(text):
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
    )
    return len(encoded["input_ids"])

model_token_counts = [
    count_model_tokens(text)
    for text in texts
]

max_tokens = max(model_token_counts)
over_limit = [
    (all_chunks[i]["chunk_id"], model_token_counts[i])
    for i in range(len(all_chunks))
    if model_token_counts[i] > model.max_seq_length
]

print("Maximum chunk tokens:", max_tokens)
print("Model max tokens:", model.max_seq_length)
print("Chunks over model limit:", len(over_limit))

if over_limit:
    print("First over-limit chunks:", over_limit[:5])

Maximum chunk tokens: 1767
Model max tokens: 512
Chunks over model limit: 17
First over-limit chunks: [('chunk_caa8e96656b9b198', 1008), ('chunk_2e64532c4bd1b03e', 954), ('chunk_d68d3a5dcc2da2a4', 1767), ('chunk_2144bbff7778e649', 1149), ('chunk_ddd8170037e54f90', 1012)]


In [16]:
# Embed all chunks.
#
# SentenceTransformer internally performs:
# text → tokenizer → token IDs → Transformer → pooling → dense vector

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    convert_to_numpy=True,
)

embeddings = np.asarray(embeddings, dtype="float32")

print("Embedding shape:", embeddings.shape)
print("dtype:", embeddings.dtype)

Batches: 100%|██████████| 36/36 [00:51<00:00,  1.43s/it]


Embedding shape: (1141, 384)
dtype: float32


In [17]:
# Build FAISS index.
#
# With normalized vectors, inner product = cosine similarity.

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS vectors:", index.ntotal)
print("Vector dimension:", dimension)

FAISS vectors: 1141
Vector dimension: 384


In [18]:
# Save index + metadata separately.
#
# FAISS stores vectors.
# JSON stores the traceability metadata needed to interpret results.

index_path = VECTOR_DIR / "quant_research.faiss"
metadata_path = VECTOR_DIR / "quant_research_metadata.json"

faiss.write_index(index, str(index_path))

metadata = []

for i, chunk in enumerate(all_chunks):
    metadata.append({
        "vector_id": i,
        "chunk_id": chunk["chunk_id"],
        "document_id": chunk["document_id"],
        "source_file": chunk.get("source_file"),
        "chunk_index": chunk.get("chunk_index"),
        "chunk_type": chunk.get("chunk_type"),
        "text": chunk.get("text"),
        "retrieval_text": chunk.get("retrieval_text"),
        "token_count": model_token_counts[i],
        "section_path": chunk.get("section_path", []),
        "pages": chunk.get("pages", []),
        "element_ids": chunk.get("element_ids", []),
        "content_types": chunk.get("content_types", []),
        "figure_ids": chunk.get("figure_ids", []),
        "table_ids": chunk.get("table_ids", []),
        "formula_ids": chunk.get("formula_ids", []),
        "image_paths": chunk.get("image_paths", []),
        "evidence": chunk.get("evidence", []),
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Index:", index_path)
print("Metadata:", metadata_path)

Index: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\vector_store\quant_research.faiss
Metadata: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\vector_store\quant_research_metadata.json


In [19]:
# Test semantic search
query = "How does the paper evaluate portfolio risk?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    convert_to_numpy=True,
).astype("float32")

k = 5
scores, ids = index.search(query_embedding, k)

for rank, (score, vector_id) in enumerate(
    zip(scores[0], ids[0]), start=1
):
    item = metadata[vector_id]

    print(f"\nRank {rank} | Score: {score:.4f}")
    print("Chunk:", item["chunk_id"])
    print("Section:", item["section_path"])
    print("Pages:", item["pages"])
    print("Text:", item["text"][:500])


Rank 1 | Score: 0.7403
Chunk: chunk_ac287c2d7ee92229
Section: ['5.3.1 Training Data (Estimation Period)']
Pages: [14]
Text: The estimated betas represent the portfolio's historical style and systematic risk. These coefficients describe the riskiness of the optimized portfolio. By estimating models on the training sample only, we avoid contamination of out-of-sample test.

Rank 2 | Score: 0.7348
Chunk: chunk_dfab370bc4b0321a
Section: ['D. Expanding-Window Walk-Forward Procedure']
Pages: [4]
Text: This process repeats for K steps, covering the entire validation or test period. The output windows are non-overlapping, so each step provides an independent out-of-sample performance estimate. For each step k, the portfolio weights produce a daily returns series r(k) t = (w(k) )⊤R(k) t where R(k) t are the asset returns on day t. Transaction costs are incorporated into the evaluation: at each rebalancing, we compute the total cost using the bid-ask spread (BA spread) of each asset on the firs

In [20]:
# Reload test — confirms the vector store is persistent.

loaded_index = faiss.read_index(str(index_path))

with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)

print("Reloaded vectors:", loaded_index.ntotal)
print("Reloaded metadata:", len(loaded_metadata))

assert loaded_index.ntotal == len(loaded_metadata) == len(all_chunks)

Reloaded vectors: 1141
Reloaded metadata: 1141


## Notebook 04 contract

Notebook 04 will use:

- `quant_research.faiss` → dense retrieval
- `quant_research_metadata.json` → chunk + source metadata
- `retrieval_text` → lexical/BM25 retrieval
- `chunk_id` → result identity
- `section_path` → structure-aware filtering
- `pages`, `element_ids`, `evidence` → citation and traceability
- `image_paths` → figure evidence

Then:

`Dense Retrieval + BM25 → Hybrid Retrieval → RRF`